# Feature Engineering — Experiences Dataset

Transforma o dataset de candidatos num dataset de **experiências isoladas** — uma linha por posição de trabalho.

| Coluna | Descrição |
|---|---|
| `experience_id` | UUID único por experiência |
| `candidate_id` | ID do candidato de origem |
| `experience` | Dict com `company`, `job_title`, `details`, `start_date`, `end_date` |
| `embedded_content` | Embedding 3072-dim do texto da experiência |

## 0. Setup

In [ ]:
!pip install -q -U google-genai pydantic python-dotenv openai requests pandas

In [ ]:
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    !git clone https://github.com/Willblemos2/agent_workshop.git
    repo_root = Path('/content/agent_workshop')
    sys.path.insert(0, str(repo_root))
else:
    repo_root = Path(os.getcwd()).parents[1]
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

DATA_DIR = repo_root / 'data'
DATA_DIR.mkdir(exist_ok=True)
print(f'repo_root : {repo_root}')
print(f'data dir  : {DATA_DIR}')

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
else:
    from src.shared.config import load_env
    load_env()

## 1. Carregar Dataset Processado

In [ ]:
from src.shared.dataset import load_processed_candidates

candidates_df = load_processed_candidates()

print(f'Candidatos : {len(candidates_df)}')
print(f'Colunas    : {list(candidates_df.columns)}')

# Quantos têm full_experience preenchido
has_exp = candidates_df['full_experience'].apply(lambda x: len(x) > 0).sum()
print(f'Com experiências: {has_exp}/{len(candidates_df)}')

## 2. Explodir — Uma Linha por Experiência

In [ ]:
import uuid
import pandas as pd

# Explode: cada experiência vira uma linha
experiences_df = (
    candidates_df[['id', 'full_experience']]
    .explode('full_experience')
    .dropna(subset=['full_experience'])
    .reset_index(drop=True)
    .rename(columns={'id': 'candidate_id', 'full_experience': 'experience'})
)

# Gera UUID único por experiência
experiences_df.insert(0, 'experience_id', [str(uuid.uuid4()) for _ in range(len(experiences_df))])

print(f'Total de experiências : {len(experiences_df)}')
print(f'Média por candidato   : {len(experiences_df) / len(candidates_df):.1f}')
print(f'Colunas               : {list(experiences_df.columns)}')
experiences_df.head(3)

In [ ]:
# Inspecionar uma experiência
experiences_df['experience'].iloc[0]

## 3. Gerar Texto para Embedding

Cada experiência é serializada num texto rico para o modelo de embedding capturar semântica de cargo, empresa e responsabilidades.

In [ ]:
def experience_to_text(exp: dict) -> str:
    end = exp.get('end_date') or 'Current'
    return (
        f"{exp['job_title']} at {exp['company']} "
        f"({exp['start_date']} – {end}). "
        f"{exp['details']}"
    )

# Prévia
sample_exp = experiences_df['experience'].iloc[0]
print(experience_to_text(sample_exp))

## 4. Embedding das Experiências

Processa em chunks com progress. A coluna resultante é `embedded_content` (3072-dim).

> **Custo estimado com text-embedding-3-large:** ~$0,13 / 1.000 experiências

In [ ]:
from src.shared.factory import build_embedding_provider

embedding_provider = build_embedding_provider()
print(f'Provider: {type(embedding_provider).__name__}')

In [ ]:
import time

CHUNK_SIZE = 50  # reduza para 10 se atingir rate limit

texts = [experience_to_text(exp) for exp in experiences_df['experience']]
total = len(texts)
all_embeddings = []
start = time.time()

for i in range(0, total, CHUNK_SIZE):
    chunk = texts[i:i + CHUNK_SIZE]
    vecs  = embedding_provider.embed_batch(chunk)
    all_embeddings.extend(vecs)

    done    = min(i + CHUNK_SIZE, total)
    elapsed = time.time() - start
    eta     = (elapsed / done) * (total - done)
    print(f'[{done}/{total} | {done/total*100:.1f}%] eta={eta/60:.1f}m  dim={len(vecs[0])}')

experiences_df['embedded_content'] = all_embeddings
print(f'\n[DONE] {total} embeddings — dim={len(all_embeddings[0])}')

In [ ]:
# Dataset final
print(f'Shape  : {experiences_df.shape}')
print(f'Colunas: {list(experiences_df.columns)}')
experiences_df[['experience_id', 'candidate_id', 'experience']].head(3)

## 5. Salvar em data/

In [ ]:
import json

output_path = DATA_DIR / 'experiences.json'

records = experiences_df.to_dict(orient='records')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f'Salvo em  : {output_path}')
print(f'Registros : {len(records)}')
print(f'Colunas   : {list(experiences_df.columns)}')